In [13]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [14]:
from pathlib import Path
import re
from collections import Counter, defaultdict
from typing import Dict, List

import pandas as pd


# ---------- paths ----------
try:
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    BASE_DIR = Path.cwd()

PROCESSED_DIR = BASE_DIR / "data" / "processed"
ALL_JOBS_PATH = PROCESSED_DIR / "all_jobs_tagged.csv"
OUT_PATH = PROCESSED_DIR / "job_roles_clean.csv"


# ---------- helpers ----------

ROLE_MAP: Dict[str, str] = {
    # interns / generic SE
    "intern": "INTERN_SE",
    "trainee": "INTERN_SE",
    "graduate": "INTERN_SE",

    # generic software engineer (junior)
    "junior software engineer": "JR_SE",
    "associate software engineer": "JR_SE",
    "software engineer": "JR_SE",

    # frontend
    "frontend developer": "JR_FE_DEV",
    "front end developer": "JR_FE_DEV",
    "front-end developer": "JR_FE_DEV",
    "frontend engineer": "JR_FE_DEV",
    "react developer": "JR_FE_DEV",
    "angular developer": "JR_FE_DEV",

    # backend
    "backend developer": "JR_BE_DEV",
    "back end developer": "JR_BE_DEV",
    "back-end developer": "JR_BE_DEV",
    "backend engineer": "JR_BE_DEV",
    "api developer": "JR_BE_DEV",

    # full-stack
    "full stack developer": "JR_FS_DEV",
    "fullstack developer": "JR_FS_DEV",
    "full-stack developer": "JR_FS_DEV",
    "full stack engineer": "JR_FS_DEV",

    # mobile
    "android developer": "JR_MOBILE_DEV",
    "android engineer": "JR_MOBILE_DEV",
    "mobile developer": "JR_MOBILE_DEV",
    "mobile app developer": "JR_MOBILE_DEV",
    "ios developer": "JR_MOBILE_DEV",
    "ios engineer": "JR_MOBILE_DEV",

    # qa / testing
    "qa engineer": "JR_QA_ENG",
    "quality assurance": "JR_QA_ENG",
    "test engineer": "JR_QA_ENG",
    "software tester": "JR_QA_ENG",
    "qa analyst": "JR_QA_ENG",

    # devops / cloud / sre
    "devops": "DEVOPS_TRAINEE",
    "site reliability": "DEVOPS_TRAINEE",
    "sre": "DEVOPS_TRAINEE",
    "cloud engineer": "DEVOPS_TRAINEE",
    "cloud devops": "DEVOPS_TRAINEE",

    # data / analytics
    "data analyst": "DATA_ANALYST_INT",
    "business intelligence": "DATA_ANALYST_INT",
    "bi analyst": "DATA_ANALYST_INT",

    "data engineer": "DATA_ENGINEER_INT",
    "etl developer": "DATA_ENGINEER_INT",
    "big data engineer": "DATA_ENGINEER_INT",

    # ai / ml
    "machine learning engineer": "AI_ML_ENGINEER_INT",
    "ml engineer": "AI_ML_ENGINEER_INT",
    "ai engineer": "AI_ML_ENGINEER_INT",
    "ai ml engineer": "AI_ML_ENGINEER_INT",

    # it support / sysadmin
    "it support": "JR_IT_SUPPORT",
    "technical support": "JR_IT_SUPPORT",
    "helpdesk": "JR_IT_SUPPORT",
    "service desk": "JR_IT_SUPPORT",
    "support engineer": "JR_IT_SUPPORT",

    "system administrator": "JR_SYS_ADMIN",
    "systems administrator": "JR_SYS_ADMIN",
    "sysadmin": "JR_SYS_ADMIN",
    "systems engineer": "JR_SYS_ADMIN",

    # business / analysis
    "business analyst": "JR_BUSINESS_ANALYST",
    "ba analyst": "JR_BUSINESS_ANALYST",

    # ui/ux / design
    "ui ux designer": "JR_UI_UX_DESIGNER",
    "ui/ux designer": "JR_UI_UX_DESIGNER",
    "ux designer": "JR_UI_UX_DESIGNER",
    "ui designer": "JR_UI_UX_DESIGNER",
}


ROLE_TITLES: Dict[str, str] = {
    "INTERN_SE": "Software Engineering Intern",
    "JR_SE": "Junior Software Engineer",
    "JR_FE_DEV": "Junior Frontend Developer",
    "JR_BE_DEV": "Junior Backend Developer",
    "JR_FS_DEV": "Junior Full-Stack Developer",
    "JR_MOBILE_DEV": "Junior Mobile Developer",
    "JR_QA_ENG": "QA Engineer (Entry Level)",
    "DEVOPS_TRAINEE": "DevOps / Cloud Trainee",
    "DATA_ANALYST_INT": "Data Analyst (Entry Level)",
    "DATA_ENGINEER_INT": "Data Engineer (Entry Level)",
    "AI_ML_ENGINEER_INT": "AI / ML Engineer (Entry Level)",
    "JR_IT_SUPPORT": "IT Support / Helpdesk (Entry Level)",
    "JR_SYS_ADMIN": "System Administrator (Entry Level)",
    "JR_BUSINESS_ANALYST": "Business Analyst (Entry Level)",
    "JR_UI_UX_DESIGNER": "UI/UX Designer (Entry Level)",
}


def normalize_title(title: str) -> str:
    if not isinstance(title, str):
        return ""
    t = title.lower()
    t = re.sub(r"[^a-z0-9+./ ]+", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()


def infer_role_id(norm_title: str) -> str:
    for pat, rid in ROLE_MAP.items():
        if pat in norm_title:
            return rid
    return ""


def infer_domain(role_id: str) -> str:
    if role_id in {"INTERN_SE", "JR_SE"}:
        return "software_engineering"
    if role_id == "JR_FE_DEV":
        return "frontend"
    if role_id == "JR_BE_DEV":
        return "backend"
    if role_id == "JR_FS_DEV":
        return "fullstack"
    if role_id == "JR_MOBILE_DEV":
        return "mobile"
    if role_id == "JR_QA_ENG":
        return "qa"
    if role_id == "DEVOPS_TRAINEE":
        return "devops"
    if role_id in {"DATA_ANALYST_INT", "DATA_ENGINEER_INT", "AI_ML_ENGINEER_INT"}:
        return "data"
    if role_id == "JR_IT_SUPPORT":
        return "it_support"
    if role_id == "JR_SYS_ADMIN":
        return "infrastructure"
    if role_id == "JR_BUSINESS_ANALYST":
        return "business_analysis"
    if role_id == "JR_UI_UX_DESIGNER":
        return "design"
    return "software_engineering"


def infer_seniority(role_id: str) -> str:
    if "INTERN" in role_id or "TRAINEE" in role_id:
        return "intern"
    return "junior"


def main():
    if not ALL_JOBS_PATH.exists():
        raise FileNotFoundError(f"{ALL_JOBS_PATH} not found")

    df = pd.read_csv(ALL_JOBS_PATH, low_memory=False)

    # candidate title columns that may exist from different sources
    candidate_cols = [c for c in ["job_name", "Job Title", "title", "name"] if c in df.columns]
    if not candidate_cols:
        raise ValueError("No title-like columns found in all_jobs_tagged.csv")

    print("Using these columns to derive titles per row:", candidate_cols)

    # choose a title per row: first non-empty from candidate_cols
    def choose_title(row):
        for c in candidate_cols:
            val = row.get(c, "")
            if isinstance(val, str) and val.strip():
                return val
        return ""

    df["title_raw"] = df.apply(choose_title, axis=1)
    df["title_norm"] = df["title_raw"].astype(str).apply(normalize_title)
    df["role_id"] = df["title_norm"].apply(infer_role_id)

    df_roles = df[df["role_id"] != ""].copy()
    print(f"Found {len(df_roles)} rows with a recognized entry-level role")

    if df_roles.empty:
        print("No roles mapped – you may need to expand ROLE_MAP patterns.")
        return

    def parse_ids(s: str) -> List[str]:
        if not isinstance(s, str):
            return []
        return [p.strip() for p in s.split(";") if p.strip()]

    df_roles["skill_id_list"] = df_roles["matched_skill_ids"].apply(parse_ids)

    role_to_skill_counts = defaultdict(Counter)
    role_to_row_count: Dict[str, int] = defaultdict(int)
    role_to_sources: Dict[str, set] = defaultdict(set)

    for _, row in df_roles.iterrows():
        rid = row["role_id"]
        role_to_row_count[rid] += 1
        src = row.get("source", "")
        if isinstance(src, str) and src:
            role_to_sources[rid].add(src)

        for sid in row["skill_id_list"]:
            role_to_skill_counts[rid][sid] += 1

    rows = []
    for rid, skill_counter in role_to_skill_counts.items():
        total_rows = role_to_row_count[rid]
        if total_rows == 0:
            continue

        skill_freq = {sid: (cnt / total_rows) for sid, cnt in skill_counter.items()}

        required_ids = [sid for sid, freq in skill_freq.items() if freq >= 0.6]
        optional_ids = [sid for sid, freq in skill_freq.items() if 0.3 <= freq < 0.6]

        required_ids.sort()
        optional_ids.sort()

        if len(required_ids) < 3:
            top_skills = [sid for sid, _ in skill_counter.most_common(5)]
            required_ids = sorted(set(required_ids + top_skills[:3]))

        domain = infer_domain(rid)
        seniority = infer_seniority(rid)
        title_norm = ROLE_TITLES.get(rid, rid)

        row = {
            "role_id": rid,
            "title_normalized": title_norm,
            "domain": domain,
            "seniority_level": seniority,
            "required_skill_ids": ";".join(required_ids),
            "optional_skill_ids": ";".join(optional_ids),
            "min_experience_years": 0,
            "max_experience_years": 2,
            "sources": ";".join(sorted(role_to_sources[rid])),
            "sample_post_count": total_rows,
        }
        rows.append(row)

    job_roles_df = pd.DataFrame(rows).sort_values("role_id").reset_index(drop=True)
    job_roles_df.to_csv(OUT_PATH, index=False)
    print(f"Saved {len(job_roles_df)} role templates to {OUT_PATH}")


if __name__ == "__main__":
    main()


Using these columns to derive titles per row: ['job_name', 'Job Title', 'title', 'name']
Found 5492 rows with a recognized entry-level role
Saved 15 role templates to /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/processed/job_roles_clean.csv
